# Siamese Network — Contrastive Training Only

This notebook is a minimal copy of [Code/SiameseNetwork.ipynb](Code/SiameseNetwork.ipynb) restricted to the contrastive objective.

Sections:
1. Setup and imports
2. Model: ResNet50 encoder + Siamese wrapper
3. Loss: Contrastive loss (+ pairwise distance utility)
4. Dataset utilities: CSV-backed pair dataset with optional bbox cropping
5. Transforms: augmentation and normalization
6. Training loop: contrastive
7. Validation: positive/negative mean distances
8. Main: wire up data loaders, train, validate

In [7]:
# 1. Setup and imports
import os
from typing import List, Tuple

import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [9]:
# Reproducibility
torch.manual_seed(42)

# 2. Model: ResNet50 embedding + Siamese wrapper
class EmbeddingNet(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        # ResNet50 backbone
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        in_features = self.backbone.fc.in_features
        # Remove classification head
        self.backbone.fc = nn.Identity()
        # Projection head to compact embedding
        self.proj = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, embed_dim),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.proj(x)
        # L2 normalize embeddings for cosine-friendly metrics
        return F.normalize(x, p=2, dim=1)


class SiameseNet(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        self.encoder = EmbeddingNet(embed_dim=embed_dim, pretrained=pretrained)

    def forward_once(self, x):
        return self.encoder(x)

    def forward(self, x1, x2):
        z1 = self.forward_once(x1)
        z2 = self.forward_once(x2)
        return z1, z2

# 3. Loss: Contrastive + pairwise distance
class ContrastiveLoss(nn.Module):
    """
    y = 1 for similar, y = 0 for dissimilar
    """
    def __init__(self, margin=1.0, distance="euclidean"):
        super().__init__()
        self.margin = margin
        self.distance = distance

    def pair_distance(self, z1, z2):
        if self.distance == "cosine":
            # Convert similarity to distance in [0, 2]
            cos_sim = F.cosine_similarity(z1, z2)
            return 1.0 - cos_sim
        else:  # euclidean
            return torch.sqrt(torch.sum((z1 - z2) ** 2, dim=1) + 1e-9)

    def forward(self, z1, z2, y):
        d = self.pair_distance(z1, z2)
        pos = y * (d ** 2)
        neg = (1 - y) * (F.relu(self.margin - d) ** 2)
        return torch.mean(pos + neg)


def pairwise_distance(z1, z2, mode="cosine"):
    if mode == "cosine":
        # Return similarity in [-1, 1]; convert to distance in [0, 2]
        sim = F.cosine_similarity(z1, z2)
        return (1.0 - sim)  # lower is more similar
    else:
        return torch.sqrt(torch.sum((z1 - z2) ** 2, dim=1) + 1e-9)

In [12]:
# 4. Dataset utilities: CSV-backed pairs with optional bbox cropping
class PairBBoxDataset(Dataset):
    """
    Dataset that reads (img_a, img_b, label) from a DataFrame with columns:
      - a_image_path, b_image_path
      - a_xmin, a_ymin, a_xmax, a_ymax (optional)
      - b_xmin, b_ymin, b_xmax, b_ymax (optional)
      - label_same (1 for similar, 0 for dissimilar)

    Added:
      - return_paths: if True, returns (img1, img2, y, p1, p2)
    """
    def __init__(self, df: 'pd.DataFrame', root_dir: str = "", crop: bool = True, transform=None, return_paths: bool = False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.crop = crop
        self.transform = transform
        self.return_paths = return_paths

    def __len__(self):
        return len(self.df)

    def _join(self, p: str) -> str:
        return p if os.path.isabs(p) else os.path.join(self.root_dir, p)

    def _safe_crop(self, img: Image.Image, xmin, ymin, xmax, ymax) -> Image.Image:
        W, H = img.size
        try:
            xmin = max(0, int(xmin))
            ymin = max(0, int(ymin))
            xmax = min(W, int(xmax))
            ymax = min(H, int(ymax))
        except Exception:
            return img
        if xmax <= xmin or ymax <= ymin:
            return img
        return img.crop((xmin, ymin, xmax, ymax))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        p1 = self._join(row['a_image_path'])
        p2 = self._join(row['b_image_path'])
        img1 = Image.open(p1).convert("RGB")
        img2 = Image.open(p2).convert("RGB")

        if self.crop:
            for prefix in ('a', 'b'):
                xmin_key, ymin_key, xmax_key, ymax_key = f'{prefix}_xmin', f'{prefix}_ymin', f'{prefix}_xmax', f'{prefix}_ymax'
                if all(k in row for k in (xmin_key, ymin_key, xmax_key, ymax_key)):
                    if prefix == 'a':
                        img1 = self._safe_crop(img1, row[xmin_key], row[ymin_key], row[xmax_key], row[ymax_key])
                    else:
                        img2 = self._safe_crop(img2, row[xmin_key], row[ymin_key], row[xmax_key], row[ymax_key])

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        y = float(row['label_same']) if 'label_same' in row else 1.0
        y_tensor = torch.tensor(y, dtype=torch.float32)

        if self.return_paths:
            return img1, img2, y_tensor, p1, p2
        return img1, img2, y_tensor


def load_pair_dfs(csv_paths: List[str]) -> 'pd.DataFrame':
    dfs = []
    for p in csv_paths:
        df = pd.read_csv(p)
        dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)
    if 'split' in df_all.columns:
        df_all['split'] = df_all['split'].astype(str).str.lower()
    else:
        df_all['split'] = 'train'
    return df_all


def get_split(df: 'pd.DataFrame', split_names) -> 'pd.DataFrame':
    if isinstance(split_names, str):
        split_names = [split_names]
    names = [s.lower() for s in split_names]
    return df[df['split'].isin(names)].copy()

# 5. Transforms: augmentation + normalization
im_size = 256
train_tf = transforms.Compose([
    transforms.Resize((im_size, im_size)),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(im_size, scale=(0.9, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# 6. Training loop: contrastive
def train_contrastive(model, loader, optimizer, device, margin=1.0, distance="cosine"):
    model.train()
    criterion = ContrastiveLoss(margin=margin, distance=distance)
    total_loss = 0.0
    for img1, img2, y in loader:
        img1, img2, y = img1.to(device), img2.to(device), y.to(device)
        z1, z2 = model(img1, img2)
        loss = criterion(z1, z2, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * img1.size(0)
    return total_loss / len(loader.dataset)

# 7. Validation: quick stats of distances
@torch.no_grad()
def validate_contrastive(model, loader, device, distance="cosine"):
    model.eval()
    pos_dists, neg_dists = [], []
    for img1, img2, y in loader:
        img1, img2 = img1.to(device), img2.to(device)
        z1, z2 = model(img1, img2)
        d = pairwise_distance(z1, z2, mode=distance).cpu()
        for di, yi in zip(d, y):
            (pos_dists if yi.item()==1 else neg_dists).append(di.item())
    pos_mean = sum(pos_dists)/max(1, len(pos_dists))
    neg_mean = sum(neg_dists)/max(1, len(neg_dists))
    return {"pos_mean": pos_mean, "neg_mean": neg_mean}

# 7b. Collect per-pair distances with optional paths for CSV export
@torch.no_grad()
def compute_val_distances_with_paths(model, loader, device, distance="cosine"):
    model.eval()
    distances, labels, paths_a, paths_b = [], [], [], []
    for batch in loader:
        # Support both (img1, img2, y) and (img1, img2, y, p1, p2)
        if len(batch) == 5:
            img1, img2, y, p1, p2 = batch
        else:
            img1, img2, y = batch
            p1 = [""] * len(y)
            p2 = [""] * len(y)

        img1, img2 = img1.to(device), img2.to(device)
        z1, z2 = model(img1, img2)
        d = pairwise_distance(z1, z2, mode=distance).cpu().tolist()
        distances.extend(d)
        labels.extend([float(yi) for yi in y])

        # p1, p2 are typically lists of strings after default collate
        try:
            paths_a.extend(list(p1))
            paths_b.extend(list(p2))
        except Exception:
            paths_a.extend([""] * len(d))
            paths_b.extend([""] * len(d))
    return distances, labels, paths_a, paths_b

# 7c. Threshold calibration: maximize F1 over a grid
def calibrate_threshold(distances, labels, thresholds=None):
    # labels expected as 0/1
    labels01 = [int(round(l)) for l in labels]
    n = len(labels01)
    if n == 0:
        raise ValueError("No validation labels provided for calibration.")

    if thresholds is None:
        # Cosine distance in [0,2], but most separation is typically within [0,1]
        step = 0.005
        thresholds = [round(i*step, 3) for i in range(int(1.0/step)+1)]  # 0.000 .. 1.000

    best = None
    metrics_list = []

    for t in thresholds:
        preds = [1 if d <= t else 0 for d in distances]
        tp = sum(1 for p,l in zip(preds, labels01) if p==1 and l==1)
        fp = sum(1 for p,l in zip(preds, labels01) if p==1 and l==0)
        tn = sum(1 for p,l in zip(preds, labels01) if p==0 and l==0)
        fn = sum(1 for p,l in zip(preds, labels01) if p==0 and l==1)

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = (2*prec*rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        acc  = (tp + tn) / n if n > 0 else 0.0

        cur = {"threshold": t, "f1": f1, "precision": prec, "recall": rec, "accuracy": acc,
               "tp": tp, "fp": fp, "tn": tn, "fn": fn}
        metrics_list.append(cur)

        if best is None:
            best = cur
        else:
            # Prefer higher F1; break ties with higher accuracy
            if (f1 > best["f1"]) or (f1 == best["f1"] and acc > best["accuracy"]):
                best = cur

    return best["threshold"], best, metrics_list

# 7d. Save per-pair predictions to CSV
def save_predictions_csv(paths_a, paths_b, distances, labels, threshold, out_path):
    df = pd.DataFrame({
        "a_image_path": paths_a,
        "b_image_path": paths_b,
        "distance": distances,
        "label_true": [int(round(l)) for l in labels] if labels is not None else [None]*len(distances),
    })
    df["pred"] = (df["distance"] <= threshold).astype(int)
    df["decision"] = df["pred"].map({1: "similar", 0: "different"})
    df["threshold"] = threshold
    df.to_csv(out_path, index=False)
    return out_path

# 8. Main: wire up everything for contrastive training + calibration + CSV export
def main():
    embed_dim = 256
    distance = "cosine"  # or "euclidean"
    margin = 1.0
    lr = 1e-4
    batch_size = 16
    epochs = 10

    # Load annotated pairs from CSVs
    csvs = [
        "data/TAMPAR/pairs_tampar_ssl.csv",
        "data/kaggle/pairs_kaggle_ssl.csv",
    ]
    df_all = load_pair_dfs(csvs)

    # Build splits from CSV "split" column
    train_df = get_split(df_all, "train")
    val_df = get_split(df_all, ["validation", "valid", "val"])
    if len(val_df) == 0:
        val_df = get_split(df_all, "validation")

    print(f"Pairs: total={len(df_all)} | train={len(train_df)} | val={len(val_df)}")

    # Datasets with bbox cropping (if bbox columns present)
    train_ds = PairBBoxDataset(train_df, transform=train_tf, crop=True, return_paths=False)
    val_ds   = PairBBoxDataset(val_df,   transform=eval_tf,  crop=True, return_paths=False)

    # Use num_workers=0 to avoid multiprocessing issues while debugging dependency resolution
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    model = SiameseNet(embed_dim=embed_dim, pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    for ep in range(epochs):
        train_loss = train_contrastive(model, train_loader, optimizer, device, margin=margin, distance=distance)
        val_stats = validate_contrastive(model, val_loader, device, distance=distance)
        print(f"Epoch {ep+1}/{epochs} | train_loss={train_loss:.4f} | val pos_mean={val_stats['pos_mean']:.3f} | val neg_mean={val_stats['neg_mean']:.3f}")

    # Threshold calibration on validation + CSV export
    # Rebuild a val loader that returns paths for CSV
    val_ds_pred = PairBBoxDataset(val_df, transform=eval_tf, crop=True, return_paths=True)
    val_loader_pred = DataLoader(val_ds_pred, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    dists, y_true, pa, pb = compute_val_distances_with_paths(model, val_loader_pred, device, distance=distance)
    # Grid in [0,1] with 0.001 step (suited for cosine distance)
    thr_grid = [i/1000 for i in range(0, 1001)]
    best_thr, best_metrics, metrics_list = calibrate_threshold(dists, y_true, thresholds=thr_grid)

    # Write calibration curve and per-pair predictions
    os.makedirs("outputs", exist_ok=True)
    metrics_csv = os.path.join("outputs", "threshold_calibration.csv")
    pd.DataFrame(metrics_list).to_csv(metrics_csv, index=False)

    preds_csv = os.path.join("outputs", "val_pair_predictions.csv")
    save_predictions_csv(pa, pb, dists, y_true, best_thr, preds_csv)

    # Summary
    n_sim = sum(1 for d in dists if d <= best_thr)
    n_diff = len(dists) - n_sim
    print(f"Calibrated threshold={best_thr:.3f} | F1={best_metrics['f1']:.3f} | Acc={best_metrics['accuracy']:.3f}")
    print(f"Predicted similar: {n_sim} | different: {n_diff} out of {len(dists)} pairs")
    print(f"Wrote per-pair predictions to {preds_csv}")
    print(f"Wrote calibration metrics to {metrics_csv}")

if __name__ == "__main__":
    main()

Pairs: total=2806 | train=1320 | val=516
Epoch 1/10 | train_loss=0.1272 | val pos_mean=-0.000 | val neg_mean=0.213
Epoch 2/10 | train_loss=0.0704 | val pos_mean=-0.000 | val neg_mean=0.229
Epoch 3/10 | train_loss=0.0487 | val pos_mean=-0.000 | val neg_mean=0.240
Epoch 4/10 | train_loss=0.0352 | val pos_mean=-0.000 | val neg_mean=0.196
Epoch 5/10 | train_loss=0.0281 | val pos_mean=-0.000 | val neg_mean=0.237
Epoch 6/10 | train_loss=0.0197 | val pos_mean=-0.000 | val neg_mean=0.292
Epoch 7/10 | train_loss=0.0155 | val pos_mean=-0.000 | val neg_mean=0.202
Epoch 8/10 | train_loss=0.0118 | val pos_mean=-0.000 | val neg_mean=0.218
Epoch 9/10 | train_loss=0.0103 | val pos_mean=-0.000 | val neg_mean=0.229
Epoch 10/10 | train_loss=0.0089 | val pos_mean=-0.000 | val neg_mean=0.191
Calibrated threshold=0.001 | F1=1.000 | Acc=1.000
Predicted similar: 258 | different: 258 out of 516 pairs
Wrote per-pair predictions to outputs/val_pair_predictions.csv
Wrote calibration metrics to outputs/threshold_c